# Multi layer perceptron regression

This notebook will cover how we trained and validated a regression model to predict student scores

The dataset contains two CSV files
 - one has the maths scores of students
 - the other has the portuguese scores of students

To start, we will analyse both separately.
Start by reading in the files into pandas dataframes:

In [3]:
import pandas as pd

df_maths = pd.read_csv("student-mat.csv", sep=";")

df_portuguese = pd.read_csv("student-por.csv", sep=";")


Then process the data (I created a library file preprocess.py to eliminate duplicated code between notebooks)

I used 10-fold cross validation with MSE as the scoring metric

In [4]:
# Data preprocessing & training
from sklearn.pipeline import Pipeline
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import KFold, cross_val_score
from numpy import mean

import preprocess
import importlib

importlib.reload(preprocess)

from preprocess import preprocess

def process_train_eval(df):
    X, y, preprocessor = preprocess(df, False)

    #Create the MLP, with two hidden layers of sizes 64 and 32 respectively. 
    mlp = MLPRegressor(hidden_layer_sizes=(64,32),activation="relu",solver="adam",alpha=1e-3,learning_rate_init=1e-3,
                    max_iter=1000,early_stopping=True,validation_fraction=0.15,n_iter_no_change=20,random_state=42)

    #Put the pre-processor and MLP into a pipeline
    model = Pipeline([("prep",preprocessor),("mlp",mlp)])


    kfold = KFold(n_splits=10, shuffle=True, random_state=42)
    scores = cross_val_score(model, X, y, cv=kfold, scoring="neg_mean_squared_error")

    print(scores)
    print(scores.std())
    print(mean(scores))


print("maths")
process_train_eval(df_maths)

print("portuguese")
process_train_eval(df_portuguese)

maths
[-18.3457252  -15.13657589 -20.6113647  -19.2289015  -19.97690484
 -13.88921392 -19.21561578 -28.30887296 -20.61046444 -12.48067106]
4.199907584566067
-18.78043102892584
portuguese
[-13.04013163  -9.63082954  -9.71116114  -5.81142185 -12.0129779
 -12.35371323  -6.08769199  -8.53510159  -6.02558531  -6.04240475]
2.719465059960538
-8.925101893935125
